In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

In [ ]:
gold_table = 'pokemons.gold.pokemon_overview'

dbutils.widgets.removeAll()
dbutils.widgets.text('process_date', datetime.now().strftime('%Y-%m-%d'), 'Processing date')
process_date = dbutils.widgets.get('process_date')

# Fail fast on a malformed date: it is used in replaceWhere and in partition filters.
datetime.strptime(process_date, '%Y-%m-%d')

# Silver keeps one snapshot per date; gold is built from the processed date only.
def silver_snapshot(table):
    return spark.table(table).filter(F.col('ingestion_date') == process_date)

silver_pokemon_table = silver_snapshot('pokemons.silver.pokemon')
silver_pokemon_stats_table = silver_snapshot('pokemons.silver.pokemon_stats')
silver_pokemon_types_table = silver_snapshot('pokemons.silver.types')

print(f"Pokémon: {silver_pokemon_table.count()} registros")
print(f"Tipos: {silver_pokemon_types_table.count()} registros")
print(f"Stats: {silver_pokemon_stats_table.count()} registros")


In [ ]:
stats_pivot = silver_pokemon_stats_table \
    .groupBy('pokemon_id') \
    .pivot('stat_name') \
    .agg(F.first('base_stat_value'))

primary_type = silver_pokemon_types_table \
    .filter(F.col('type_slot') == 1) \
    .select(
        'pokemon_id',
        F.col('type_name').alias('primary_type')
    )

df_overview = silver_pokemon_table.alias("p") \
    .join(primary_type.alias("pt"), "pokemon_id", "left") \
    .join(stats_pivot.alias("s"), "pokemon_id", "left") \
    .select(
        F.col("p.pokemon_id"),
        F.col("p.name_clean").alias("name"),
        F.col("pt.primary_type"),
        F.coalesce(F.col("p.height_cm"), F.lit(0)).cast("int").alias("height_cm"),
        F.coalesce(F.col("p.weight_kg"), F.lit(0.0)).alias("weight_kg"),
        F.coalesce(F.col("p.base_xp"), F.lit(0)).alias("base_xp"),
        # Stats individuais
        F.coalesce(F.col("s.hp"), F.lit(0)).alias("hp"),
        F.coalesce(F.col("s.attack"), F.lit(0)).alias("attack"),
        F.coalesce(F.col("s.defense"), F.lit(0)).alias("defense"),
        F.coalesce(F.col("s.`special-attack`"), F.lit(0)).alias("special_attack"),
        F.coalesce(F.col("s.`special-defense`"), F.lit(0)).alias("special_defense"),
        F.coalesce(F.col("s.speed"), F.lit(0)).alias("speed"),
        # MÉTRICA DE NEGÓCIO: poder total
        (
            F.coalesce(F.col("s.hp"), F.lit(0)) +
            F.coalesce(F.col("s.attack"), F.lit(0)) +
            F.coalesce(F.col("s.defense"), F.lit(0)) +
            F.coalesce(F.col("s.`special-attack`"), F.lit(0)) +
            F.coalesce(F.col("s.`special-defense`"), F.lit(0)) +
            F.coalesce(F.col("s.speed"), F.lit(0))
        ).alias("total_power")
    )



In [0]:
df_overview.write \
    .format('delta') \
    .mode('overwrite') \
    .saveAsTable(gold_table)

spark.sql('SELECT * FROM pokemons.gold.pokemon_overview LIMIT 5').display()

pokemon_id,name,primary_type,height_cm,weight_kg,base_xp,hp,attack,defense,special_attack,special_defense,speed,total_power
1,bulbasaur,grass,7,69,64,45,49,49,65,65,45,318
2,ivysaur,grass,10,130,142,60,62,63,80,80,60,405
3,venusaur,grass,20,1000,236,80,82,83,100,100,80,525
4,charmander,fire,6,85,62,39,52,43,60,50,65,309
5,charmeleon,fire,11,190,142,58,64,58,80,65,80,405


In [0]:
df_type_ranking = spark.table(gold_table) \
    .groupBy('primary_type') \
    .agg(
        F.count('*').alias('pokemon_count'),
        F.round(F.avg('total_power'), 1).alias('average_total_power'),
        F.round(F.avg("attack"), 1).alias("avg_attack"),
        F.round(F.avg("defense"), 1).alias("avg_defense"),
        F.round(F.avg("hp"), 1).alias("avg_hp"),
        F.round(F.avg("speed"), 1).alias("avg_speed")
    ) \
    .orderBy(F.desc('average_total_power'))

display(df_type_ranking)
    

primary_type,pokemon_count,average_total_power,avg_attack,avg_defense,avg_hp,avg_speed
ice,2,517.5,67.5,67.5,77.5,90.0
psychic,8,470.1,60.1,57.5,63.9,93.0
fire,12,455.6,83.9,62.6,63.8,84.0
electric,9,445.6,62.0,64.7,54.4,100.0
dragon,3,440.0,94.0,68.3,64.3,66.7
rock,9,420.6,82.2,110.0,53.9,58.3
fighting,7,412.1,102.9,61.0,63.6,66.1
water,28,411.2,70.3,77.5,64.5,67.7
grass,12,410.3,70.7,69.6,65.0,52.1
ghost,3,405.0,50.0,45.0,45.0,95.0


In [0]:
df_type_ranking.write \
    .format('delta') \
    .mode('overwrite') \
    .saveAsTable('pokemons.gold.type_power_ranking')
    
spark.sql('SELECT * FROM pokemons.gold.type_power_ranking LIMIT 5').display()

primary_type,pokemon_count,average_total_power,avg_attack,avg_defense,avg_hp,avg_speed
ice,2,517.5,67.5,67.5,77.5,90.0
psychic,8,470.1,60.1,57.5,63.9,93.0
fire,12,455.6,83.9,62.6,63.8,84.0
electric,9,445.6,62.0,64.7,54.4,100.0
dragon,3,440.0,94.0,68.3,64.3,66.7


In [ ]:
w = Window.orderBy(F.desc('total_power'))

df_tier_list = spark.table(gold_table) \
    .withColumn('rank_overall', F.row_number().over(w)) \
    .withColumn('tier_num', F.ntile(4).over(w)) \
    .withColumn('tier_label', 
                F.when(F.col('tier_num') == 1, 'S')
                .when(F.col('tier_num') == 2, 'A')
                .when(F.col('tier_num') == 3, 'B')
                .otherwise('C')
    ) \
    .select(
        'pokemon_id',
        'name',
        'primary_type',
        'total_power',
        'rank_overall',
        'tier_label'
    ) \
    .orderBy('rank_overall')

display(df_tier_list.limit(20))


In [0]:
df_tier_list.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("pokemons.gold.pokemon_tier_list")

spark.sql("SELECT * FROM pokemons.gold.pokemon_tier_list where tier_label = 'S' LIMIT 10").display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


pokemon_id,name,primary_type,total_power,rank_overall,tier_label
150,mewtwo,psychic,680,1,S
149,dragonite,dragon,600,2,S
151,mew,psychic,600,3,S
144,articuno,ice,580,4,S
145,zapdos,electric,580,5,S
146,moltres,fire,580,6,S
59,arcanine,fire,555,7,S
130,gyarados,water,540,8,S
143,snorlax,normal,540,9,S
131,lapras,water,535,10,S


In [0]:


spark.sql("""
    SELECT
        (SELECT COUNT(*) FROM pokemons.gold.pokemon_overview) as overview,
        (SELECT COUNT(*) FROM pokemons.gold.type_power_ranking) as types,
        (SELECT COUNT(*) FROM pokemons.gold.pokemon_tier_list) as ranking
""").display()

print("\n TOP 5 MAIS FORTES:")
spark.sql("""
    SELECT name, primary_type, total_power, tier_label
    FROM pokemons.gold.pokemon_tier_list 
    ORDER BY total_power DESC LIMIT 5
""").display()

print("\nTIPOS MAIS FORTES:")
spark.sql("""
    SELECT primary_type, average_total_power, pokemon_count
    FROM pokemons.gold.type_power_ranking 
    ORDER BY average_total_power DESC LIMIT 5
""").display()




overview,types,ranking
151,15,151



 TOP 5 MAIS FORTES:


name,primary_type,total_power,tier_label
mewtwo,psychic,680,S
dragonite,dragon,600,S
mew,psychic,600,S
articuno,ice,580,S
zapdos,electric,580,S



TIPOS MAIS FORTES:


primary_type,average_total_power,pokemon_count
ice,517.5,2
psychic,470.1,8
fire,455.6,12
electric,445.6,9
dragon,440.0,3
